# Project 5 — Natural Language Processing

**Author:** Vernon T. Cox · Data Science Student, CNM · GitHub: santed7

This notebook solves all three parts of the NLP project:

1. **Part 1** — From a CSV of famous people, pick a *reference person* and output the 10 people whose overview text is "closest" in an NLP sense. Also report the reference person's sentiment.
2. **Part 2** — Use the Wikipedia API to pull the *full* articles for the reference person and those 10 neighbors, re-rank the neighbors by similarity on the full articles, and compare (plot) the two rankings.
3. **Part 3** — Make the notebook interactive: type or choose a name and see the 10 closest people.

*Comments are written so that someone reading this cold in six months — including me — can follow what each cell does, why it exists, and what output to expect.*


## AI Assistance Statement

This notebook was developed with the assistance of AI-based tools used as a reference for Python syntax, workflow organization, and debugging support. AI helped identify issues related to missing values, dataset preparation, and proper sequencing of NLP steps such as text cleaning, vectorization, and similarity ranking. All code included in this notebook was reviewed, executed, and validated to ensure it performs the intended analysis, and I maintained responsibility for understanding the process and interpreting the results.


## Setup — installs and imports

In [1]:
%%capture
# ONE-TIME SETUP per Colab session (re-run after a fresh runtime / "Restart runtime").
#   textblob      -> sentiment analysis (Parts 1 and 2)
#   wikipedia-api -> pull full Wikipedia article text (Part 2)
#   ipywidgets    -> the dropdown / search box in Part 3 (usually preinstalled in Colab)
# The %%capture line above hides the long, noisy pip log so the cell output stays tidy.
!pip install -q -U textblob wikipedia-api ipywidgets
!python -m textblob.download_corpora


In [2]:
# Imports (run once per session).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from textblob import TextBlob                                 # sentiment analysis
from sklearn.feature_extraction.text import TfidfVectorizer   # text -> number vectors
from sklearn.neighbors import NearestNeighbors                # find the closest overviews

import wikipediaapi                                           # Part 2: full article text
from IPython.display import display, clear_output             # nice tables + widget refresh

# Show all columns of wide DataFrames instead of truncating them with "...".
pd.options.display.max_columns = 100


## Settings

All the knobs live here in one place (named constants instead of magic numbers scattered through the code). Change values **here**, not in the cells below.


In [3]:
# --- Project settings (edit these, not the code below) ---

# Where the data lives (the class-provided CSV of famous people).
CSV_URL = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv"

# The DEFAULT reference person. You no longer edit a name here -- when you run
# the notebook a prompt asks you for a name (see "Choose the reference person"
# below). This default is used only if you press Enter at that prompt without
# typing anything. It MUST match a value in the CSV's 'name' column exactly
# (names come from DBpedia, e.g. "Barack Obama").
DEFAULT_NAME = "Barack Obama"

# How many similar people to return (the assignment asks for 10).
N_NEIGHBORS = 10

# How many characters of the Wikipedia article to PREVIEW in Part 2. Full articles
# are long; printing all of one floods the output, so we print just this much.
WIKI_PREVIEW_CHARS = 2000

# Wikipedia's API policy REQUIRES a user-agent string identifying the project and
# a contact. Edit this to your own details if you like.
WIKI_USER_AGENT = "cnm-ds-project5-nlp (student project; contact: santed7 on GitHub)"


# Part 1 — CSV data: nearest people + sentiment

**Goal:** pick a reference person, output the 10 people whose overview text is closest (NLP-wise), and report the reference person's sentiment.


### Load the data

In [4]:
# Read the CSV directly from the URL (no manual download needed).
# Columns are:
#   URI  -> DBpedia link for the person (we mostly ignore it)
#   name -> the person's name (what we search and match on)
#   text -> a short overview paragraph about the person (what NLP compares)
# EXPECT roughly 42,785 rows.
people = pd.read_csv(CSV_URL)
print("rows, columns:", people.shape)
people.head()


rows, columns: (42786, 3)


,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...


In [5]:
# Quick health check: column names, dtypes, and missing-value counts.
# We mainly care that 'text' has few or no missing entries.
people.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42786 entries, 0 to 42785
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   URI     42786 non-null  object
 1   name    42786 non-null  object
 2   text    42786 non-null  object
dtypes: object(3)
memory usage: 1002.9+ KB


In [6]:
# We cannot measure similarity on an empty overview, so drop rows whose 'text'
# is missing. reset_index(drop=True) renumbers the rows 0,1,2,... so each row's
# POSITION lines up exactly with its row in the TF-IDF matrix later. (This avoids
# a subtle bug where neighbor positions point at the wrong people.)
before = len(people)
people = people.dropna(subset=["text"]).reset_index(drop=True)
print("dropped", before - len(people), "rows with no overview text")
print("rows remaining:", len(people))


dropped 0 rows with no overview text
rows remaining: 42786


### Choose the reference person (type a name)

Run the next cell and type any person's name at the prompt. It resolves what
you type to an exact name in the data, so `obama` or `barack obama` both work.
Press **Enter** with nothing typed to fall back to the default in Settings.

In [9]:
# Pick the reference person at RUN TIME instead of hard-coding it in Settings.
#   what:   prompt for a name, then resolve it to an EXACT name in the data.
#   why:    so you can analyze ANYONE in the CSV without editing code first.
#   expect: prints the resolved name (e.g. you type "obama" -> "Barack Obama").
# COLAB NOTE: the input box appears at the TOP of this cell's output area, not
# the bottom -- look up there if you do not see it right away.

# Helper: turn whatever was typed into an exact name from the 'name' column.
#   1) exact match              -> use as-is
#   2) same name, any case      -> use the data's official spelling
#   3) part of a name, 1 hit    -> use that one match
#   4) part of a name, many hits-> show candidates, ask to re-run more specific
#   5) nothing matches          -> clear "not found" error
# Single-shot by design (no input loop), so it never blocks the cells below.
def resolve_name(typed):
    typed = typed.strip()

    # 1) Exact match on the official spelling.
    if typed in set(people["name"]):
        return typed

    # 2) Case-insensitive exact match (handles "barack obama").
    ci_exact = people.loc[people["name"].str.lower() == typed.lower(), "name"].tolist()
    if len(ci_exact) >= 1:
        return ci_exact[0]

    # 3) Partial match anywhere in the name (handles "obama").
    partial = people.loc[
        people["name"].str.contains(typed, case=False, na=False), "name"
    ].tolist()
    if len(partial) == 1:
        return partial[0]
    if len(partial) > 1:
        shown = ", ".join(partial[:10])
        raise ValueError(
            "'" + typed + "' matched " + str(len(partial)) + " names, e.g. "
            + shown + ". Re-run this cell and type the exact name you want."
        )

    # 4) No match at all.
    raise ValueError(
        "'" + typed + "' was not found in the data. Re-run this cell and check "
        "the spelling (the search box in Part 3 also lists every name)."
    )

# Prompt once. Empty input falls back to the default set in Settings.
typed_name = input("Enter a person's name (Enter for default '" + DEFAULT_NAME + "'): ")
if typed_name.strip() == "":
    typed_name = DEFAULT_NAME

# This becomes the single name the WHOLE notebook is built around, downstream.
REFERENCE_NAME = resolve_name(typed_name)
print("Reference person set to:", REFERENCE_NAME)


Enter a person's name (Enter for default 'Barack Obama'): Ronald Reagan


ValueError: 'Ronald Reagan' was not found in the data. Re-run this cell and check the spelling (the search box in Part 3 also lists every name).

### Confirm the reference person is in the data

In [ ]:
# Find the row POSITION of the reference person by an exact name match. We use
# this position to pull their TF-IDF vector out of the matrix below.
# If the name is not found, raise a clear error now instead of crashing later in
# a more confusing place.
matches = people.index[people["name"] == REFERENCE_NAME].tolist()
if len(matches) == 0:
    raise ValueError(
        "'" + REFERENCE_NAME + "' was not found in the data. "
        "Check spelling/capitalization, or run the partial-search cell below to "
        "find the exact name, then paste it into REFERENCE_NAME in Settings."
    )
ref_idx = matches[0]
print("Reference person:", REFERENCE_NAME)
print("Found at row position:", ref_idx)


In [ ]:
# Optional manual helper: list names CONTAINING a string (case-insensitive).
# The name prompt above already resolves partial names for you, but you can use
# this to browse the data directly. Change "Obama" to whoever you want to find.
# na=False keeps any blank/non-string names from breaking the search.
people[people["name"].str.contains("Obama", case=False, na=False)]


### Vectorize the overviews with TF-IDF

In [ ]:
# Turn every overview into a numeric vector with TF-IDF.
#   - TF-IDF gives high weight to words that are DISTINCTIVE to a person and low
#     weight to words everyone uses, so "closeness" reflects meaningful overlap
#     rather than shared filler words.
#   - stop_words="english" drops common filler ("the", "and", "of", ...).
#   - max_features caps the vocabulary at the 20,000 most informative words so the
#     matrix stays a manageable size on 40k+ documents.
# EXPECT a sparse matrix shaped (number_of_people, vocabulary_size).
vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)
tfidf = vectorizer.fit_transform(people["text"])
print("TF-IDF matrix shape (people, words):", tfidf.shape)


### Find the 10 nearest people

In [ ]:
# Build a NearestNeighbors index over the TF-IDF vectors.
#   metric="cosine" compares the ANGLE between two text vectors, which is the
#   standard, length-independent way to measure document similarity (a long and a
#   short overview can still be "close" in topic).
# Fitting just stores the vectors for fast lookup; no real model training happens.
nn = NearestNeighbors(metric="cosine").fit(tfidf)
print("NearestNeighbors index ready.")


In [ ]:
# Ask for the reference person's nearest neighbors.
# We request N_NEIGHBORS + 1 because the closest match is always the person
# THEMSELVES (distance ~0); we drop that so we return 10 OTHER people.
#   distances -> cosine distance to each neighbor (0 = identical, 1 = unrelated)
#   indices   -> row positions of those neighbors in `people`
distances, indices = nn.kneighbors(tfidf[ref_idx], n_neighbors=N_NEIGHBORS + 1)

# kneighbors returns 2-D arrays (one row per query); we queried one person, so
# flatten to plain 1-D arrays for easy handling.
distances = distances[0]
indices = indices[0]

# Drop the reference person themselves (the first entry, distance ~0).
neighbor_idx = indices[1:]
neighbor_dist = distances[1:]
print("Neighbor row positions:", neighbor_idx)


In [ ]:
# Build a tidy table of the 10 closest people, nearest first.
#   rank     -> 1..10 (1 = most similar)
#   distance -> cosine distance (smaller = more similar)
# This table is the Part 1 deliverable: the 10 overviews closest to the reference.
part1_results = pd.DataFrame({
    "rank": range(1, N_NEIGHBORS + 1),
    "name": people.loc[neighbor_idx, "name"].values,
    "distance": neighbor_dist.round(4),
})
part1_results


### Sentiment of the reference person's overview

In [ ]:
# TextBlob sentiment gives two numbers:
#   polarity     -> -1.0 (negative) ... +1.0 (positive)
#   subjectivity ->  0.0 (factual)  ...  1.0 (opinionated)
# Encyclopedia-style overviews are usually fairly neutral and factual, so EXPECT a
# polarity near 0 and a low-to-moderate subjectivity.
ref_text = people.loc[ref_idx, "text"]
ref_sentiment = TextBlob(ref_text).sentiment

print("Sentiment of", REFERENCE_NAME, "overview:")
print("  polarity:    ", round(ref_sentiment.polarity, 4))
print("  subjectivity:", round(ref_sentiment.subjectivity, 4))


# Part 2 — Full Wikipedia articles: re-rank and compare

**Goal:** pull the *full* Wikipedia article for the reference person and each of the 10 neighbors, re-rank the neighbors by similarity on the full articles, then compare that ranking to the Part 1 ranking (which used only the short overviews). The difference in rank is the comparison.


### Set up the Wikipedia client

In [ ]:
# Create the Wikipedia API client. The user_agent is REQUIRED by Wikipedia's
# policy; language="en" selects English Wikipedia.
wiki = wikipediaapi.Wikipedia(user_agent=WIKI_USER_AGENT, language="en")

def get_wiki_text(name):
    # Return the full plain text of a person's English Wikipedia page, or an empty
    # string "" if no exact-title page exists. We look the page up by the person's
    # name; if Wikipedia has no matching title, page.exists() is False.
    # Returning "" (instead of raising) lets the rest of the code handle a missing
    # page gracefully.
    page = wiki.page(name)
    if page.exists():
        return page.text
    return ""


### Step 1 — print the reference person's Wikipedia text

In [ ]:
# Pull the reference person's FULL Wikipedia article.
ref_wiki_text = get_wiki_text(REFERENCE_NAME)

# Articles are long, so print only the first WIKI_PREVIEW_CHARS as a sanity check.
# (Change WIKI_PREVIEW_CHARS in Settings to preview more or less.)
print("Article length (characters):", len(ref_wiki_text))
print("-" * 60)
print(ref_wiki_text[:WIKI_PREVIEW_CHARS])


### Step 2 — sentiment of the full Wikipedia page

In [ ]:
# Same TextBlob sentiment as Part 1, but on the FULL article instead of the short
# overview. EXPECT it to still be fairly neutral, though it may differ a little from
# the overview's sentiment because there is far more text to average over.
ref_wiki_sentiment = TextBlob(ref_wiki_text).sentiment
print("Sentiment of", REFERENCE_NAME, "full Wikipedia page:")
print("  polarity:    ", round(ref_wiki_sentiment.polarity, 4))
print("  subjectivity:", round(ref_wiki_sentiment.subjectivity, 4))


### Step 3 — collect the 10 neighbors' Wikipedia articles

In [ ]:
# Download the FULL article for each of the 10 neighbors, keeping the same order
# as the Part 1 ranking. NOTE: this makes ~10 live web requests, so it takes a few
# seconds. A name with no exact Wikipedia title comes back as "" and is flagged
# below (article chars: 0).
neighbor_names = people.loc[neighbor_idx, "name"].tolist()

neighbor_wiki_texts = []
for person_name in neighbor_names:
    article = get_wiki_text(person_name)
    neighbor_wiki_texts.append(article)
    print(person_name, "-> article chars:", len(article))


### Step 4 — re-rank the neighbors using the full articles

In [ ]:
# Re-measure similarity on the FULL articles instead of the short overviews.
# Strategy: vectorize [reference_article] + [the 10 neighbor articles] together with
# a fresh TF-IDF, then measure each neighbor's cosine distance to the reference,
# which sits at row 0.
#
# We build the document list with the reference FIRST, then the 10 neighbors, so
# neighbor i lives at document row i+1.
wiki_docs = [ref_wiki_text] + neighbor_wiki_texts

# Fresh vectorizer for this small corpus (11 documents); no max_features needed.
# NOTE: any neighbor whose page was missing ("") becomes an all-zero vector and
# will land at the maximum distance (1.0) -- effectively ranked last, which is the
# sensible behavior for "we could not read their page."
wiki_vectorizer = TfidfVectorizer(stop_words="english")
wiki_tfidf = wiki_vectorizer.fit_transform(wiki_docs)

# Cosine distance from the reference (row 0) to every document.
wiki_nn = NearestNeighbors(metric="cosine").fit(wiki_tfidf)
w_dist, w_idx = wiki_nn.kneighbors(wiki_tfidf[0], n_neighbors=len(wiki_docs))
w_dist = w_dist[0]
w_idx = w_idx[0]

# Map results back to names. Row 0 is the reference itself, so skip it. Neighbor
# articles are rows 1..10 in the SAME order as neighbor_names, so the name for
# document row d is neighbor_names[d - 1].
wiki_ranking = []
for position, doc_index in enumerate(w_idx):
    if doc_index == 0:
        continue
    name = neighbor_names[doc_index - 1]
    wiki_ranking.append((name, round(w_dist[position], 4)))

# Table ranked by full-article similarity (nearest first).
part2_results = pd.DataFrame(wiki_ranking, columns=["name", "wiki_distance"])
part2_results["wiki_rank"] = range(1, len(part2_results) + 1)
part2_results


### Step 5 — compare the two rankings

In [ ]:
# Put the two rankings side by side, matched on name:
#   overview_rank -> Part 1 rank (short overview)
#   wiki_rank     -> Part 2 rank (full Wikipedia article)
compare = part1_results[["name", "rank"]].rename(columns={"rank": "overview_rank"})
compare = compare.merge(
    part2_results[["name", "wiki_rank"]],
    on="name",
    how="left",   # keep all 10 overview neighbors even if a wiki page was missing
)

# rank_difference = how far each person moved between the two methods.
#   0     -> same rank both ways
#   large -> the full article told a different story than the short overview
# .abs() so a move of +3 and -3 both count as a difference of 3.
compare["rank_difference"] = (compare["overview_rank"] - compare["wiki_rank"]).abs()
compare


In [ ]:
# Plot the comparison: one point per person.
#   x = Part 1 rank (overview),  y = Part 2 rank (full article)
# Points ON the dashed diagonal ranked the SAME both ways; points far OFF it changed
# a lot between the two methods.
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(compare["overview_rank"], compare["wiki_rank"], s=80)

# Label each point with the person's name (skip any with a missing wiki rank).
for _, row in compare.iterrows():
    if pd.notna(row["wiki_rank"]):
        ax.annotate(row["name"],
                    (row["overview_rank"], row["wiki_rank"]),
                    fontsize=8, xytext=(5, 5), textcoords="offset points")

# Diagonal reference line y = x (same rank in both methods).
ax.plot([1, N_NEIGHBORS], [1, N_NEIGHBORS], "r--", label="same rank")
ax.set_xlabel("Part 1 rank (short overview)")
ax.set_ylabel("Part 2 rank (full Wikipedia article)")
ax.set_title("Overview ranking vs full-article ranking for " + REFERENCE_NAME)
ax.legend()
plt.show()


# Part 3 — Interactive: type or choose a name, get the 10 closest people

We wrap the Part 1 logic in a reusable function, then offer two ways to use it: a plain `input()` prompt that works everywhere, and an `ipywidgets` dropdown + search box.


In [ ]:
# Reusable function: given a name, return the 10 closest people as a DataFrame.
# This is just the Part 1 logic packaged so we can call it for ANY name on demand.
# Returns None (and prints a hint) if the name is not found.
def find_closest_people(name, n=N_NEIGHBORS):
    # Locate the person's row position by exact name match.
    hits = people.index[people["name"] == name].tolist()
    if len(hits) == 0:
        print("'" + name + "' not found. Check spelling, or use the search box below.")
        return None
    idx = hits[0]

    # Nearest neighbors (n + 1 so we can drop the person themselves).
    dist, ind = nn.kneighbors(tfidf[idx], n_neighbors=n + 1)
    ind = ind[0][1:]      # drop self (first result)
    dist = dist[0][1:]

    return pd.DataFrame({
        "rank": range(1, n + 1),
        "name": people.loc[ind, "name"].values,
        "distance": dist.round(4),
    })

# Quick test on the reference person.
find_closest_people(REFERENCE_NAME)


### Option A — `input()` version (works everywhere)

In [ ]:
# Option A -- simplest interactive form: type a name, get the 10 closest people.
# Run this cell, type a name at the prompt, and press Enter.
# COLAB NOTE: the input box appears at the TOP of this cell's output area, not at
# the bottom -- look up there if you do not see it right away.
typed_name = input("Enter a famous person's name: ")
display(find_closest_people(typed_name))


### Option B — widget version (dropdown + search box)

In [ ]:
# Option B -- richer interactive form using ipywidgets:
#   1) Type part of a name in the Search box to filter the dropdown.
#   2) Pick a person from the dropdown; the 10 closest update automatically.
# If widgets do not render in your environment, use Option A above instead.
import ipywidgets as widgets

# A text box to filter names, a dropdown of matches, and an output area.
search_box = widgets.Text(description="Search:", placeholder="type part of a name")
name_dropdown = widgets.Dropdown(description="Person:", options=[])
out = widgets.Output()

# A sorted list of all names, used to populate the dropdown.
ALL_NAMES = sorted(people["name"].dropna().unique().tolist())

def update_dropdown(change):
    # Filter names by the search text (case-insensitive). Cap at 200 options so the
    # dropdown stays responsive on 40k+ names.
    query = search_box.value.strip().lower()
    if query == "":
        name_dropdown.options = ALL_NAMES[:200]
    else:
        name_dropdown.options = [nm for nm in ALL_NAMES if query in nm.lower()][:200]

def show_results(change):
    # When a dropdown selection is made, show that person's 10 closest people.
    with out:
        clear_output()
        if name_dropdown.value:
            display(find_closest_people(name_dropdown.value))

# Wire up the events: typing filters the dropdown; selecting shows results.
search_box.observe(update_dropdown, names="value")
name_dropdown.observe(show_results, names="value")

# Initialize the dropdown contents and show the widgets.
update_dropdown(None)
display(search_box, name_dropdown, out)


---
### Presentation note

For the live demo, classmates will suggest a famous person who exists in the DBpedia set. Type their name into the search box (Option B) or run Option A and type it at the prompt — the notebook returns the 10 closest individuals on the spot.
